# Session 5: Building from scratch (II) — Robust modeling and evaluation

Previously...

<p style="font-size:120%;font-style:italic">• Given an ECG signal, develop a method to classify each beat by its origin in the cardiac tissue</p>

Then we explored, understood, cleaned and homogeneized the data.

And when we tried a first simple machine learning model:

<table>
    <tr>
        <td style="padding-right: 30pt;">
            <table>
                <thead>
                    <tr>
                        <th></th>
                        <th>Pred 0</th>
                        <th>Pred 1</th>
                        <th>Pred 2</th>
                        <th>Pred 3</th>
                    </tr>
                </thead>
                <tbody>
                    <tr>
                        <td>True 0</td>
                        <td>2920</td>
                        <td>0</td>
                        <td>6</td>
                        <td>0</td>
                    </tr>
                    <tr>
                        <td>True 1</td>
                        <td>51</td>
                        <td>57</td>
                        <td>0</td>
                        <td>0</td>
                    </tr>
                    <tr>
                        <td>True 2</td>
                        <td>44</td>
                        <td>0</td>
                        <td>182</td>
                        <td>3</td>
                    </tr>
                    <tr>
                        <td>True 3</td>
                        <td>6</td>
                        <td>0</td>
                        <td>3</td>
                        <td>10</td>
                    </tr>
                </tbody>
            </table>
        </td>
        <td>
            <table>
                <thead>
                    <tr>
                        <th></th>
                        <th>Precision</th>
                        <th>Recall</th>
                        <th>F1-Score</th>
                        <th>Support</th>
                    </tr>
                </thead>
                <tbody>
                    <tr>
                        <td>0</td>
                        <td>0.97</td>
                        <td>1.00</td>
                        <td>0.98</td>
                        <td>2926.00</td>
                    </tr>
                    <tr>
                        <td>1</td>
                        <td>1.00</td>
                        <td>0.53</td>
                        <td>0.69</td>
                        <td>108.00</td>
                    </tr>
                    <tr>
                        <td>2</td>
                        <td>0.95</td>
                        <td>0.79</td>
                        <td>0.87</td>
                        <td>229.00</td>
                    </tr>
                    <tr>
                        <td>3</td>
                        <td>0.77</td>
                        <td>0.53</td>
                        <td>0.62</td>
                        <td>19.00</td>
                    </tr>
                    <tr>
                        <td>Accuracy</td>
                        <td>0.97</td>
                        <td>0.97</td>
                        <td>0.97</td>
                        <td>0.97</td>
                    </tr>
                    <tr>
                        <td>Macro Avg</td>
                        <td>0.92</td>
                        <td>0.71</td>
                        <td>0.79</td>
                        <td>3282.00</td>
                    </tr>
                    <tr>
                        <td>Weighted Avg</td>
                        <td>0.97</td>
                        <td>0.97</td>
                        <td>0.96</td>
                        <td>3282.00</td>
                    </tr>
                </tbody>
            </table>
        </td>
    </tr>
</table>

<p style="font-size:400%;color:darkred;text-align:center">🥳</p>

---- 
<p style="font-size:400%;color:darkred;text-align:center">⚠️⚠️ WRONG!!! ⚠️⚠️</p>

The last, quick experiment with a simple ML model is **fundamentally wrong**, and any conclusion we may get from it will be misleading!

### **What happened?**

 - After we segmented and homogeneized the raw signal for each individual heartbeat, we created a big `X` matrix with one row per beat, and proceeded in the standard way:

<pre>
<code class="language-python">
# Prepare the X and y matrices
X = np.hstack(df_sample2.beat_sig).T
X = X - np.median(X, axis=1, keepdims=True)
y = np.repeat(df_sample2.type, 2)
# Split in training and test
<span style="background-color: #ffff99;">X_train, X_test, y_train, y_test = <a href="https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html">train_test_split</a>(X, y, test_size=0.2, random_state=42)</span>
# Create the model, train it and test it
classifier = RandomForestClassifier()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
</code>
</pre>

However, the core of any Machine Learning methodology is the assumption that **`X_train` and `X_test` are independent and indentically distributed**. We have violated the independence assumption in two different ways:
 - By including beats **from the same subject** in training and testing.
 - By including **the same beat** (with different projections corresponding to the leads) in training and testing.

This problem, commonly know as **data leakage**, is pervasive in real studies using ML, and the main cause of overestimated results, even in peer-reviewed publications.
 - Example: [Ankışhan et.al: *Early stage lung cancer detection from speech sounds in natural environments* (2024)](https://tomas-teijeiro.github.io/Slides_BCAM/#/5/5).

Let's recall the procedure for addressing any realistic problem with Machine Learning:
 1. Learning about the problem to be solved.
 2. Understanding the available data.
 3. Data cleaning, preparation and homogeneization.
 4. **Design of experimental validation.**
 5. Choose a model or set of models to evaluate.
 6. Training and evaluating the model(s).
 7. Extract conclusions, assess the limitations of the model.

The problem with step 4 is that it is totally dependent on the **specific problem** to be solved, and using the same strategy for two problems that in principle look similar may be catastrophic. 
 - For example, mixing data from the same patient in training and testing can be perfectly fine, and even required, if we are building a personalized model.

## Getting back to the end of Session 4

The following code cells load and prepare the data as we did it in the last session:

In [2]:
import os
os.environ["KERAS_BACKEND"] = "jax"
#os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

#General imports
import collections
import numpy as np
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
import keras
#Figure settings to avoid super large plots
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 90
#Local path to the MIT-BIH Arrhythmia
MITDB = '/opt/tljh/common/mitdb'

In [3]:
#Global structures with target labels, mapping to classes, and list of recordings to analyze
BEAT_CODES = set(['N','L','R','B','A','a','J','S','V','F','e','j','n','E','/','f','Q'])
LABEL_MAP = {'N':0,'L':0,'R':0,'B':0,
             'A':1,'a':1,'J':1,'S':1,'e':1,'j':1,'n':1,
             'V':2,'E':2,
             'F':3,
             '/':4,'f':4,'Q':4}
REC_LIST = [r.strip() for r in open(f'{MITDB}/RECORDS', 'r').readlines()]

In Session 4, we agreed that we should remove leads `V5`, `V2` and `V4` due to the extremely small number of samples compared with the other leads. However, if we are doing the split at patient level, we need to consider all existing **lead combinations**. Let's get a quick summary:

In [4]:
collections.Counter([tuple(wfdb.rdrecord(f'{MITDB}/{r}').sig_name) for r in REC_LIST])

Counter({('MLII', 'V1'): 40,
         ('MLII', 'V5'): 2,
         ('V5', 'V2'): 2,
         ('MLII', 'V2'): 2,
         ('V5', 'MLII'): 1,
         ('MLII', 'V4'): 1})

It seems evident we should select the recordings with leads `(MLII, V1)`. Let's filter them:

In [5]:
REC_LIST = [r for r in REC_LIST if tuple(wfdb.rdrecord(f'{MITDB}/{r}').sig_name)==('MLII', 'V1')]
print(REC_LIST, len(REC_LIST))

['101', '105', '106', '107', '108', '109', '111', '112', '113', '115', '116', '118', '119', '121', '122', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214', '215', '217', '219', '220', '221', '222', '223', '228', '230', '231', '232', '233', '234'] 40


In [6]:
#Code from the last session to load and organize every beat in a global dataframe
def adjust_array_length(arr, peak):
    """Utility function to adjust the length of each array"""
    desired_length = 1501
    target_peak_index = 750
    current_length = len(arr)

    start_index = max(0, peak - target_peak_index)
    end_index = min(current_length, peak + (desired_length - target_peak_index))

    # Truncate the array around the peak
    truncated_array = arr[start_index:end_index, :]

    left_padding = target_peak_index - (peak - start_index)
    right_padding = desired_length - len(truncated_array) - left_padding

    # Extend the array if needed
    if left_padding > 0:
        left_pad_value = truncated_array[0, :]
        left_extension = np.full((left_padding, 2), left_pad_value)
    else:
        left_extension = np.empty(shape=(0, 2))

    if right_padding > 0:
        right_pad_value = truncated_array[-1, :]
        right_extension = np.full((right_padding, 2), right_pad_value)
    else:
        right_extension = np.empty(shape=(0, 2))

    # Combine the extensions and truncated array
    adjusted_array = np.concatenate((left_extension, truncated_array, right_extension))

    return adjusted_array
    
# Creation of a Pandas dataframe with homogeneized data for every beat
full_data = []
offset = 18 #Window cut with respect to previous and next peak.
for r in REC_LIST:
    rec = wfdb.rdrecord(f'{MITDB}/{r}')
    anns = wfdb.rdann(f'{MITDB}/{r}', 'atr')
    beat_mask = np.array([s in BEAT_CODES for s in anns.symbol])
    beat_indices = np.where(beat_mask)[0]
    beat_types = np.array(anns.symbol)[beat_indices]
    beat_locations = anns.sample[beat_indices]
    for i in range(1, len(beat_indices)-1):
        beat_data = {}
        bsig = rec.p_signal[beat_locations[i-1]+offset:beat_locations[i+1]-offset,:]
        bsig = bsig - np.median(bsig, axis=0, keepdims=True)
        beat_data['beat_sig'] = adjust_array_length(bsig, 
                        beat_locations[i]-beat_locations[i-1]-offset)
        beat_data['type'] = LABEL_MAP[beat_types[i]]
        beat_data['leads'] =  rec.sig_name
        beat_data['recname'] = rec.record_name
        full_data.append(beat_data)
df_beats = pd.DataFrame(full_data)

We also concluded that class 4 should be removed from the analysis, basically for the very same reason that we removed some of the leads:

In [7]:
df_beats = df_beats.query('type!=4')

As a general recommendation, while you are developing and exploring initial models, do it with a restricted amount of data to speed-up things:

In [8]:
df_sample = df_beats.sample(n=10000, random_state=42)

#### 📋 Exercise 0: Quantifying the leakage effect

Before fixing the methodology, let's measure the size of the bias. In order to guarantee the independence of training and test sets, we will make the split ensuring no data from the same recording are in training and testing. The [`sklearn.model_selection.GroupShuffleSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html) splitter is specifically designed for these use cases.

 - Run two experiments side by side **on the same data and the same model**, differing only in the splitter:

1. `train_test_split(...)` — the old, leaky split.
2. `GroupShuffleSplit(...)` grouped by `recname` — the correct split.

Report the weighted-F1 score in both cases. The gap between them is the size of the lie.

In [9]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
#Full sample dataset as X, y matrices:
X = np.hstack(df_sample.beat_sig).T
X = X - np.median(X, axis=1, keepdims=True)
y = np.repeat(df_sample.type, 2).values

##### Leaky split:

In [9]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Create the model, train it and test it
classifier = RandomForestClassifier()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
# Display the confusion matrix and the classification report
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=[f'True {l}' for l in labels], columns=[f'Pred {l}' for l in labels])
display(cm_df)
display(pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).transpose().round(2))

,Pred 0,Pred 1,Pred 2,Pred 3
True 0,3462,1,6,0
True 1,64,66,4,0
True 2,66,0,293,1
True 3,10,0,4,23


,precision,recall,f1-score,support
0,0.96,1.00,0.98,3469.00
1,0.99,0.49,0.66,134.00
2,0.95,0.81,0.88,360.00
3,0.96,0.62,0.75,37.00
accuracy,0.96,0.96,0.96,0.96
macro avg,0.96,0.73,0.82,4000.00
weighted avg,0.96,0.96,0.96,4000.00


##### Correct data split by subject:

In [11]:
# TODO complete the code for making the split at the subject level
gsplit = GroupShuffleSplit(??)
train_idx, test_idx = next(gsplit.split(X, y, ??))
X_train, X_test, y_train, y_test = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
# Create the model, train it and test it
classifier = RandomForestClassifier()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
# Display the confusion matrix and the classification report
labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=[f'True {l}' for l in labels], columns=[f'Pred {l}' for l in labels])
display(cm_df)
display(pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).transpose().round(2))

,Pred 0,Pred 1,Pred 2
True 0,3212,0,94
True 1,386,1,5
True 2,121,3,326


,precision,recall,f1-score,support
0,0.86,0.97,0.91,3306.00
1,0.25,0.00,0.01,392.00
2,0.77,0.72,0.75,450.00
accuracy,0.85,0.85,0.85,0.85
macro avg,0.63,0.57,0.55,4148.00
weighted avg,0.80,0.85,0.81,4148.00


### 📋 Exercise 1

Define, build, train and validate a Machine Learning model to solve the heartbeat classification problem, considering all the issues we overlooked in Session 4.

 - The main metric to assess the quality of the model will be **weighted average F1 score** for the 4 classes. Please fill the table below with the results you will obtain.

__Hints:__
 - A CNN is probably the most promising model a priori. You may use a 2D CNN in case you want to input the two leads of each beat simultaneously, or a 1D CNN in case you want to use single leads.
     - Suggested architecture to get started:
         - 3 1D-Convolutional layers, with filter size 64, 128 and 256 (kernel size 3), each one followed by BatchNormalization and MaxPooling1D (with pool size 2).
         - 2 Dense layers (after a Flatten one), with 128 and 64 units. Dropout of 0.5 after each of them.
         - Output Dense layer, with 4 outputs (one per class).
 - Be extremely **careful with the data splitting** strategy. Make sure statistical independence is maintained between training, validation and test.
 - **Start from a simple** model, using a **single lead**, and get a working pipeline. Then, optimize over that baseline. Even the classifier tested in Exercise 0 could do the job for this.
 - **Class imbalance** may be a big problem when training. Consider playing with the `class_weight` and `sample_weight` parameters of `model.fit()`.
     - Accuracy is usually not a good metric to follow in imbalanced scenarios.
 - **Rhythm information** (distance to the next and previous beats) is essential for a proper detection of some classes. While this information is already in the data format we developed, it may be difficult to extract automatically. Explicitly providing these values as features can help.
 - The ideal validation pipeline for this kind of problems with limited amount of subjects is **leave-one-subject out cross-validation**, or at least k-fold cross-validation.

### Results Table

| Configuration                          | Splitter        | Channels | Rhythm feats | Weighted F1 |
|----------------------------------------|-----------------|----------|--------------|-------------|
| Baseline (Session 4)                   | `train_test`    | 2        | No           | ~0.96 (fake)|
| Subject-independent baseline           | `GroupShuffle`  | 2        | No           | ~0.81       |
| Single-lead CNN model                  | `GroupShuffle`  | 1        | No           |             |
| CNN + rhythm features                  | `GroupShuffle`  | 1        | Yes          |             |
| Group 5-fold CV (mean ± std)           | `GroupKFold`    | 1        | Yes          |             |

### Some utility code you may find helpful


Here you can find some utility code to move forward:
 1. How to properly manage the ECG channels as an additional matrix dimension.
 2. Convert the training and testing matrices to single-channel ECGs.
 3. A baseline CNN architecture.
 4. A small feature matrix with specific information about rhythm. This can boost performance for some classes.
    - To include both the raw signals and the rhythm features in the model, use two `Input` layers. Consider the following snippet as an example:
   
```Python
def build_two_input_model(signal_length, n_channels, n_rhythm_feats, n_classes):
    # --- Branch 1: raw signal ---
    sig_in = keras.Input(shape=(signal_length, n_channels), name="signal")
    x = keras.layers.Conv1D(32, 15, padding="same", activation="relu")(sig_in)
    x = keras.layers.MaxPool1D(2)(x)
    # ... more conv / residual blocks ...
    x = keras.layers.GlobalAveragePooling1D()(x)   # -> (batch, F_cnn)

    # --- Branch 2: rhythm features ---
    feat_in = keras.Input(shape=(n_rhythm_feats,), name="rhythm")
    f = keras.layers.BatchNormalization()(feat_in)  # or a Normalization layer adapted on train
    # optional: f = layers.Dense(16, activation="relu")(f)

    # --- Late fusion ---
    merged = keras.layers.Concatenate()([x, f])
    merged = keras.layers.Dropout(0.3)(merged)
    out = keras.layers.Dense(n_classes, activation="softmax")(merged)

    return keras.Model(inputs=[sig_in, feat_in], outputs=out, name="cnn_plus_rhythm")

#At training time...
model.fit(
    {"signal": X_sig, "rhythm": X_feat},
    ...
)
```

In [15]:
#The data matrix X has the two channels of the same beat as different samples (rows).
#We can properly manage this by adding a new dimension corresponding to each channel.
X_multi = X.reshape(X.shape[0]//2, 2, X.shape[1]).transpose(0, 2, 1)
#Labels should not be repeated 
y_multi = df_sample.type.values

#New split, as the number of total rows have changed
gsplit = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gsplit.split(X_multi, y_multi, groups=df_sample.recname.values))
X_train, X_test, y_train, y_test = X_multi[train_idx], X_multi[test_idx], y_multi[train_idx], y_multi[test_idx]

In [11]:
#Convert the data to single-channel (here we keep the first one, MLII).
X_train = X_train[:, :, :1]
X_test = X_test[:,:,:1]
X_train.shape

(7926, 1501, 1)

In [12]:
def build_cnn_baseline(inputs, n_classes):
    inp = keras.layers.Input(shape=(inputs.shape[1], inputs.shape[2]))
    x = keras.layers.Conv1D(filters=64, kernel_size=11, activation='relu')(inp)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling1D(pool_size=18)(x)
    #Second convolutional layer
    x = keras.layers.Conv1D(filters=128, kernel_size=7, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling1D(pool_size=5)(x)
    #Third convolutional layer
    x = keras.layers.Conv1D(filters=256, kernel_size=5, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling1D(pool_size=5)(x)
    #Flatten and dense layers
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dropout(0.1)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.Dropout(0.1)(x)
    #Output layer
    out = keras.layers.Dense(4, activation='softmax')(x)
    return keras.Model(inp, out, name="ecg_cnn_baseline")

model_cnn_baseline = build_cnn_baseline(X_train, 4)
model_cnn_baseline.compile(optimizer='adam',
                           loss='sparse_categorical_crossentropy',
                           metrics=['accuracy'])
model_cnn_baseline.summary()

Model: "ecg_cnn_baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1501, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 1491, 64)       │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1491, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 82, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 76, 128)        │        57,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 76, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 11, 256)        │       164,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 11, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 2, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 298,308 (1.14 MB)

 Trainable params: 297,412 (1.13 MB)

 Non-trainable params: 896 (3.50 KB)

In [24]:
#Code to create new training and testing matrices with rhythm features.
from sklearn.preprocessing import StandardScaler
rh = []
for _, beat in df_sample.iterrows():
    df_ch1 = np.diff(beat.beat_sig[:, 0])
    nz = np.where(df_ch1)[0]
    pb, nb = 750-nz[0], nz[-1]-750
    rh.append(np.array([pb, nb, nb/pb]))
rh = np.array(rh)
X_rh_train = rh[train_idx]
X_rh_test = rh[test_idx]
scaler = StandardScaler()
X_rh_train = scaler.fit_transform(X_rh_train)
X_rh_test = scaler.transform(X_rh_test)

### 📋 Exercise 1A: Leakage-free baseline CNN

Train the reference 1D-CNN architecture on a **single channel** (MLII), using a **subject-independent** train/test split. Report the weighted F1 score on the test set, broken down by class.

#### Specification

- Splitter: `GroupShuffleSplit(n_splits=1, test_size=0.2)` grouped by `recname`.
- Channels: single channel (`MLII`, i.e. `X[:, :, :1]`).
- Architecture: the `cnn_baseline` provided.
- Loss: `sparse_categorical_crossentropy`.
- Metric: track `accuracy` during training but **report weighted F1 on the test set**.
- Epochs: 20 to start. Yo may use `EarlyStopping` on validation loss with patience 5.
- Imbalance: use `class_weight='balanced'`-style weights computed from the training labels.

#### Hints

- `sklearn.utils.class_weight.compute_class_weight` computes balanced weights from labels.
- Keras expects the `class_weight` argument as a dict `{class_idx: weight}`.

### 📋 Exercise 1B: Adding rhythm features

Class 1 (supraventricular) is hard to distinguish from class 0 (normal) based on **morphology alone**, because the QRS waveforms look almost identical. The discriminating information is in the **rhythm**: a supraventricular beat is *premature* (closer to the previous beat than expected).

Use the helper code provided above to construct rhythm features `(pre_beat_dist, post_beat_dist, ratio)` for every beat. Then extend the model to use **two inputs**:

- The 1D ECG signal → goes through the conv stack.
- The 3 rhythm features → go directly to a small dense branch.

Both branches are concatenated before the classification head.

#### Specification

- Use the Keras **Functional API** (not `Sequential`).
- The rhythm branch can be as simple as `Dense(16, relu) → Dense(16, relu)`.
- Concatenate, then re-use the existing dense classification head.

#### Hints

- `keras.layers.Concatenate()([branch_a_output, branch_b_output])`
- The model now takes a **list** of arrays as input: `model.fit([X_signal, X_rhythm], y, ...)`.
- The rhythm features are already standardized in `X_rh_train` / `X_rh_test`.

### 📋 Exercise 1C: Robust evaluation via Group k-fold

A single split is statistically fragile. Let's get a confidence interval on the model's performance by running **5-fold cross-validation, grouped by recording**.

#### Specification

- Use `sklearn.model_selection.GroupKFold(n_splits=5)`.
- For each fold: train the best model achieved so far, record the weighted F1 on the held-out fold.
- Report: mean ± std of the 5 weighted-F1 scores.

#### Hints
- Training 5 models is expensive. Keep `epochs` small (10–15) and use `EarlyStopping`.
- Make sure you **reinitialize** the model inside each fold; `model.fit` does not reset the weights between calls.
- Use `keras.models.clone_model` to start from the same architecture but fresh weights.